# MobileViT Pipeline Runner
Use **Run All** to execute the project scripts in order and keep all outputs in one place.

## 0) Install dependencies
Run once per environment setup.

In [1]:
%pip install roboflow mediapipe opencv-python Pillow torch torchvision transformers onnx onnxruntime --quiet

Note: you may need to restart the kernel to use updated packages.


## 1) Download dataset
Requires `ROBOFLOW_API_KEY` env variable to be set with your personal key.

In [2]:
%run Download_dataset.py

loading Roboflow workspace...
loading Roboflow project...
Dataset downloaded successfully


## 2) Crop hands
Reads from `NinjutsuAR---4152-6/`, writes cropped 256x256 images to `dataset_cropped/`.

In [3]:
%run Crop_hands.py

Found 16122 images to process.

  [50/16122] 0.3%  |  14.3 img/s  |  ETA: 18.8 min
  [100/16122] 0.6%  |  14.5 img/s  |  ETA: 18.5 min
  [150/16122] 0.9%  |  15.2 img/s  |  ETA: 17.5 min
  [200/16122] 1.2%  |  13.7 img/s  |  ETA: 19.4 min
  [250/16122] 1.6%  |  12.1 img/s  |  ETA: 21.8 min
  [300/16122] 1.9%  |  11.3 img/s  |  ETA: 23.2 min
  [350/16122] 2.2%  |  10.8 img/s  |  ETA: 24.2 min
  [400/16122] 2.5%  |  10.6 img/s  |  ETA: 24.8 min
  [450/16122] 2.8%  |  10.4 img/s  |  ETA: 25.1 min
  [500/16122] 3.1%  |  10.3 img/s  |  ETA: 25.4 min
  [550/16122] 3.4%  |  10.2 img/s  |  ETA: 25.5 min
  [600/16122] 3.7%  |  10.4 img/s  |  ETA: 24.9 min
  [650/16122] 4.0%  |  10.3 img/s  |  ETA: 25.1 min
  [700/16122] 4.3%  |  10.5 img/s  |  ETA: 24.4 min
  [750/16122] 4.7%  |  10.9 img/s  |  ETA: 23.5 min
  [800/16122] 5.0%  |  11.2 img/s  |  ETA: 22.7 min
  [850/16122] 5.3%  |  11.5 img/s  |  ETA: 22.2 min
  [900/16122] 5.6%  |  11.7 img/s  |  ETA: 21.7 min
  [950/16122] 5.9%  |  12.1 img/s

## 3) Sanity check — data pipeline

In [4]:
%run debug.py

Image shape: torch.Size([4, 3, 256, 256])
Image min: -0.992 | max: 0.969
Labels: tensor([0, 0, 0, 0])
Classes: ['Bird', 'Boar', 'Dog', 'Dragon', 'Hare', 'HollowPurple', 'Horse', 'InfiniteVoid', 'MalevolentShrine', 'Monkey', 'Ox', 'Ram', 'Rat', 'Snake', 'Tiger']
Label distribution sample: ['Bird', 'Bird', 'Bird', 'Bird']


## 4) Sanity check — model forward pass

In [5]:
%run debug2.py

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
[transformers] You passed `num_labels=15` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 347/347 [00:00<00:00, 33269.10it/s]
[transformers] MobileViTForImageClassification LOAD REPORT from: apple/mobilevit-xx-small
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 320]) vs model:torch.Size([15, 320])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([15])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Output shape: torch.Size([4, 15])
Output sample: tensor([ 0.4768,  0.0983,  1.0648,  0.6855,  0.5569, -0.4453,  0.4257, -0.7942,
         1.1349, -0.7889,  0.2047, -0.9824,  0.5600, -0.4235,  0.4602])
Predicted classes: tensor([8, 3, 2, 3])
Actual labels: tensor([ 0,  1, 13,  1])
Classes: ['Bird', 'Boar', 'Dog', 'Dragon', 'Hare', 'HollowPurple', 'Horse', 'InfiniteVoid', 'MalevolentShrine', 'Monkey', 'Ox', 'Ram', 'Rat', 'Snake', 'Tiger']


## 5) Train MobileViT-XXS
Two-phase fine-tuning. Saves best checkpoint to `best_model.pth`.

In [6]:
%run train.py

Training on: cuda
DataLoader workers: 0 | pin_memory: True
Classes (15): ['Bird', 'Boar', 'Dog', 'Dragon', 'Hare', 'HollowPurple', 'Horse', 'InfiniteVoid', 'MalevolentShrine', 'Monkey', 'Ox', 'Ram', 'Rat', 'Snake', 'Tiger']
Train images: 8593 | Val images: 2408 | Test images: 1209


[transformers] You passed `num_labels=15` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 347/347 [00:00<00:00, 29163.88it/s]
[transformers] MobileViTForImageClassification LOAD REPORT from: apple/mobilevit-xx-small
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 320]) vs model:torch.Size([15, 320])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([15])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.



Phase 1 — Freezing backbone, training head only...
  Epoch 1/10 | Loss: 1.591 | Acc: 65.7%
  Epoch 2/10 | Loss: 1.217 | Acc: 77.7%
  Epoch 3/10 | Loss: 1.142 | Acc: 80.9%
  Epoch 4/10 | Loss: 1.112 | Acc: 82.0%
  Epoch 5/10 | Loss: 1.100 | Acc: 82.2%
  Epoch 6/10 | Loss: 1.096 | Acc: 82.6%
  Epoch 7/10 | Loss: 1.082 | Acc: 82.6%
  Epoch 8/10 | Loss: 1.075 | Acc: 83.5%
  Epoch 9/10 | Loss: 1.077 | Acc: 83.1%
  Epoch 10/10 | Loss: 1.077 | Acc: 82.9%

Phase 2 — Unfreezing all layers, full fine-tuning...
  Epoch 1/20 | Train loss 0.857 acc 92.9% | Val loss 0.780 acc 96.9% ← best saved
  Epoch 2/20 | Train loss 0.755 acc 96.8% | Val loss 0.693 acc 99.2% ← best saved
  Epoch 3/20 | Train loss 0.706 acc 98.4% | Val loss 0.670 acc 99.6% ← best saved
  Epoch 4/20 | Train loss 0.678 acc 99.0% | Val loss 0.643 acc 99.8% ← best saved
  Epoch 5/20 | Train loss 0.660 acc 99.3% | Val loss 0.633 acc 99.9% ← best saved
  Epoch 6/20 | Train loss 0.647 acc 99.5% | Val loss 0.624 acc 99.9%
  Epoch 7/20 |

## 6) Export to ONNX
Creates `mobilevit_seals.onnx` and `class_names.json`.

In [9]:
%run export.py

[transformers] You passed `num_labels=15` which is incompatible to the `id2label` map of length `1000`.


Classes: ['Bird', 'Boar', 'Dog', 'Dragon', 'Hare', 'HollowPurple', 'Horse', 'InfiniteVoid', 'MalevolentShrine', 'Monkey', 'Ox', 'Ram', 'Rat', 'Snake', 'Tiger']


Loading weights: 100%|██████████| 347/347 [00:00<00:00, 20764.47it/s]
[transformers] MobileViTForImageClassification LOAD REPORT from: apple/mobilevit-xx-small
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 320]) vs model:torch.Size([15, 320])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([15])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Exported — mobilevit_seals_v3.onnx ready
Saved class mapping → class_names.json  (15 classes)

Demo: running a dummy input through the exported ONNX model...
  Predicted index : 10
  Predicted class : Ox
  Predicted logit : 26.399408

  Top-5 classes by logit:
    1. Ox -> 26.399408
    2. Dog -> 17.872143
    3. Hare -> 5.651368
    4. MalevolentShrine -> 3.532907
    5. Horse -> -1.177088
